# Hoofdstuk 7: Chatapplicaties bouwen
## OpenAI API Quickstart

Dit notitieboek is aangepast van de [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) die notitieboeken bevat die toegang bieden tot [Azure OpenAI](aoai-assignment.ipynb) services.

De Python OpenAI API werkt ook met Azure OpenAI-modellen, met een paar aanpassingen. Lees hier meer over de verschillen: [Hoe te schakelen tussen OpenAI en Azure OpenAI eindpunten met Python](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Overzicht  
"Grote taalmodellen zijn functies die tekst naar tekst omzetten. Gegeven een invoertekst probeert een groot taalmodel de tekst die daarna zal komen te voorspellen"(1). Deze "quickstart" notebook zal gebruikers introduceren in de hoog-niveau LLM concepten, kernpakketvereisten om te beginnen met AML, een zachte introductie tot promptontwerp, en verschillende korte voorbeelden van uiteenlopende gebruiksscenario's. 


## Inhoudsopgave  

[Overzicht](#overview)  
[Hoe OpenAI Service te gebruiken](#how-to-use-openai-service)  
[1. Je OpenAI Service aanmaken](#1.-creating-your-openai-service)  
[2. Installatie](#2.-installation)    
[3. Referenties](#3.-credentials)  

[Toepassingsgevallen](#use-cases)    
[1. Tekst samenvatten](#1.-summarize-text)  
[2. Tekst classificeren](#2.-classify-text)  
[3. Nieuwe productnamen genereren](#3.-generate-new-product-names)  
[4. Een classifier fijn afstemmen](#4.fine-tune-a-classifier)  

[Referenties](#references)


### Bouw je eerste prompt  
Deze korte oefening geeft een basale introductie voor het indienen van prompts bij een OpenAI-model voor een eenvoudige taak "samenvatten".  


**Stappen**:  
1. Installeer de OpenAI-bibliotheek in je Python-omgeving  
2. Laad standaard hulpbibliotheken en stel je gebruikelijke OpenAI-beveiligingsreferenties in voor de OpenAI Service die je hebt aangemaakt  
3. Kies een model voor je taak  
4. Maak een eenvoudige prompt voor het model  
5. Dien je verzoek in bij de model-API!  


### 1. Installeer OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Importeer hulpbibliotheken en maak referenties aan


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Het juiste model vinden  
Modellen zoals GPT-4o en GPT-4o mini kunnen natuurlijke taal begrijpen en genereren, en zijn beschikbaar op het OpenAI-platform met verschillende niveaus van kracht en snelheid die geschikt zijn voor verschillende taken.


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. Promptontwerp  

"De magie van grote taalmodellen is dat door getraind te zijn om deze voorspellingsfout over enorme hoeveelheden tekst te minimaliseren, de modellen uiteindelijk concepten leren die nuttig zijn voor deze voorspellingen. Ze leren bijvoorbeeld concepten zoals"(1):

* hoe te spellen
* hoe grammatica werkt
* hoe te parafraseren
* hoe vragen te beantwoorden
* hoe een gesprek te voeren
* hoe in veel talen te schrijven
* hoe te coderen
* enzovoort.

#### Hoe een groot taalmodel te besturen  
"Van alle invoer voor een groot taalmodel is de tekstprompt verreweg de meest invloedrijke(1).

Grote taalmodellen kunnen op een paar manieren worden aangestuurd om output te produceren:

Instructie: Vertel het model wat je wilt
Voltooiing: Zet het model ertoe aan het begin van wat je wilt te voltooien
Demonstratie: Laat het model zien wat je wilt, met:
Een paar voorbeelden in de prompt
Honderden of duizenden voorbeelden in een fine-tuning trainingsdataset"



#### Er zijn drie basisrichtlijnen voor het maken van prompts:

**Toon en vertel**. Maak duidelijk wat je wilt door middel van instructies, voorbeelden, of een combinatie van beide. Als je wilt dat het model een lijst items alfabetisch rangschikt of een alinea classificeert op sentiment, laat dan zien dat dat is wat je wilt.

**Lever kwaliteitsdata**. Als je een classifier probeert te bouwen of het model een patroon wilt laten volgen, zorg dan dat er voldoende voorbeelden zijn. Controleer je voorbeelden zorgvuldig — het model is meestal slim genoeg om basis spellingfouten te doorzien en je toch een antwoord te geven, maar het kan ook aannemen dat dit opzettelijk is, wat de reactie kan beïnvloeden.

**Controleer je instellingen.** De temperature- en top_p-instellingen bepalen hoe deterministisch het model een reactie genereert. Als je een antwoord vraagt waar maar één juist antwoord is, wil je deze lager instellen. Zoek je meer diverse reacties, dan kun je ze hoger zetten. De meest gemaakte fout met deze instellingen is aannemen dat het 'slimheid' of 'creativiteit'-knoppen zijn.


Bron: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Indienen!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Herhaal dezelfde oproep, hoe vergelijken de resultaten? 


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Tekst Samenvatten  
#### Uitdaging  
Vat tekst samen door aan het einde van een tekstpassage een 'tl;dr:' toe te voegen. Let op hoe het model begrijpt hoe het een aantal taken kan uitvoeren zonder extra instructies. Je kunt experimenteren met meer beschrijvende prompts dan tl;dr om het gedrag van het model aan te passen en de samenvatting die je ontvangt te personaliseren(3).  

Recente onderzoeken hebben aanzienlijke vooruitgang aangetoond bij veel NLP-taken en benchmarks door vooraf te trainen op een grote hoeveelheid tekst gevolgd door fijn afstemmen op een specifieke taak. Hoewel deze methode doorgaans taak-agnostisch is in de architectuur, vereist het nog steeds taak-specifieke fijn-afstemmingsdatasets van duizenden of tienduizenden voorbeelden. Ter vergelijking: mensen kunnen doorgaans een nieuwe taaltaak uitvoeren met slechts een paar voorbeelden of eenvoudige instructies - iets waar huidige NLP-systemen nog grotendeels moeite mee hebben. Hier tonen we aan dat het opschalen van taalmodellen de taak-agnostische few-shot prestaties sterk verbetert en soms zelfs competitief wordt met eerdere state-of-the-art fijn-afstemmingsmethoden. 



Tl;dr


# Oefeningen voor verschillende gebruikssituaties  
1. Tekst samenvatten  
2. Tekst classificeren  
3. Nieuwe productnamen genereren


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Tekst Classificeren  
#### Uitdaging  
Classificeer items in categorieën die tijdens het inferentieproces worden opgegeven. In het volgende voorbeeld geven we zowel de categorieën als de te classificeren tekst in de prompt(*playground_reference). 

Klantvraag: Hallo, een van de toetsen op mijn laptoptoetsenbord is onlangs kapotgegaan en ik heb een vervanging nodig:

Geclassificeerde categorie:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Nieuwe productnamen genereren
#### Uitdaging
Maak productnamen op basis van voorbeeldwoorden. Hier voegen we in de prompt informatie toe over het product waarvoor we namen gaan genereren. We geven ook een soortgelijk voorbeeld om het patroon te laten zien dat we willen ontvangen. We hebben ook de temperatuurwaarde hoog gezet om meer willekeur en meer innovatieve reacties te krijgen.

Productbeschrijving: Een thuis milkshake maker
Zaadwoorden: snel, gezond, compact.
Productnamen: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Productbeschrijving: Een paar schoenen dat op iedere voetmaat past.
Zaadwoorden: aanpasbaar, passend, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Referenties  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Best practices for fine-tuning GPT models to classify text](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Voor Meer Hulp  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# Bijdragers
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
